In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from bs4 import BeautifulSoup
import requests
from groq import Groq
from openai import OpenAI

In [2]:
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}
def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


In [4]:

load_dotenv(override=True)


api_key = os.getenv("GROQ_API_KEY")


if api_key and api_key.startswith("gsk_") and len(api_key) > 20:
    print("✅ API key looks good")
else:
    raise ValueError("❌ Invalid GROQ API key. Check your .env file")
client = Groq(api_key=api_key)

MODEL = "qwen/qwen3-32b"

✅ API key looks good


In [5]:
links = fetch_website_contents("https://huggingface.co")
links

'Hugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nHauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive\nUpdated\n14 days ago\n•\n326k\n•\n897\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\nabout 20 hours ago\n•\n164k\n•\n1.17k\nbaidu/Qianfan-OCR\nUpdated\n6 days ago\n•\n8.49k\n•\n333\nnvidia/Nemotron-Cascade-2-30B-A3B\nUpdated\n1 day ago\n•\n19.7k\n•\n257\nRoyalCities/Foundation-1\nUpdated\n8 days ago\n•\n248\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n1.53k\nWan2.2 14B Preview\n🐌\n1.53k\ngenerate a video from an image with a text prompt\nRunning\non\nZero\nFeatured\n167\nDLSS 5 Anything\n🎮\n167\nTurn any image into a DLSS 5 meme (using FLUX

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/login
/join
/spaces
/models
/HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
/Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
/baidu/Qianfan-OCR
/nvidia/Nemotron-Cascade-2-30B-A3B
/RoyalCities/Foundation-1
/models
/spaces/r3gm/wan2-2-fp8da-aoti-preview
/spaces/victor/dlss-5-anything
/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast
/spaces/deddytoyota/Free-Unlimited-Google-Veo-3
/spaces/FrameAI4687/Omni-Video-Factory
/spaces
/datasets/open-index/hacker-news
/datasets/OpenMOSS-Team/OmniAction
/datasets/ropedia-ai/xperience-10m
/datasets/stepfun-ai/Step-3.5-Flash-SFT
/datasets/th1nhng0/vietnamese-legal-documents
/da

In [9]:
def select_relevant_links(url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'careers page',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'company page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'home page', 'url': 'https://huggingface.co/'}]}

In [14]:
select_relevant_links("https://www.youtube.com")

{'links': [{'type': 'about page', 'url': 'https://www.youtube.com/about/'},
  {'type': 'press page', 'url': 'https://www.youtube.com/about/press/'},
  {'type': 'creators page', 'url': 'https://www.youtube.com/creators/'},
  {'type': 'ads page', 'url': 'https://www.youtube.com/ads/'},
  {'type': 'how youtube works',
   'url': 'https://www.youtube.com/howyoutubeworks?utm_campaign=ytgen&utm_source=ythp&utm_medium=LeftNav&utm_content=txt'}]}

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
14 days ago
•
326k
•
897
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
about 20 hours ago
•
164k
•
1.17k
baidu/Qianfan-OCR
Updated
6 days ago
•
8.49k
•
334
nvidia/Nemotron-Cascade-2-30B-A3B
Updated
1 day ago
•
19.7k
•
257
RoyalCities/Foundation-1
Updated
8 days ago
•
248
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.53k
Wan2.2 14B Preview
🐌
1.53k
generate a video from an image with a text prompt
Running
on
Zero
Featured
167
DLSS 5 Anything
🎮
167
Turn any image into a DLSS 5 meme (using FLUX.2-klein-9b-kv)
Running
on
Zero
MCP
Featured
441
FireRed

In [17]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nHauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive\nUpdated\n14 days ago\n•\n326k\n•\n897\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\nabout 20 hours ago\n•\n164k\n•\n1.17k\nbaidu/Qianfan-OCR\nUpdated\n6 days ago\n•\n8.49k\n•\n334\nnvidia/Nemotron-Cascade-2-30B-A3B\nUpdated\n1 day ago\n•\n19.7k\n•\n257\nRoyalCities/Foundation-1\nUpdated\n8 days ago\n•\n248\nBr

In [20]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

<think>
Alright, I need to create a brochure for Hugging Face based on the provided pages. Let's start by understanding the key elements. The landing page mentions it's a community building the future with models, datasets, and applications. They have over 2 million models and 1 million applications. The About page gives more details about collaboration and the open-source stack. The Enterprise page talks about team plans and enterprise solutions. 

First, the brochure should have a title and a welcome section. I'll mention Hugging Face as the AI community platform. Then, a section on what they offer: Models, Datasets, Spaces, Buckets, etc. Highlight the collaboration platform aspect.

Next, company culture. The landing page emphasizes community and collaboration. Mention the open-source stack and moving faster with their tools.

For customers, focus on developers, researchers, organizations. Maybe mention enterprise plans with security and support features.

Careers: The careers page is empty, so I'll just mention them being a remote-first company and possibly link to the careers page.

I need to structure each section with headings and bullet points. Make sure to use markdown without code blocks. Avoid technical jargon but keep it professional. Check for any missing info, like specific projects or partnerships, but based on the data provided, stick to what's there. Ensure the brochure is concise but covers all necessary aspects for prospective customers, investors, and recruits.
</think>

# Hugging Face Brochure  

## 🌐 **About Hugging Face**  
Hugging Face is a global AI community and collaboration platform where developers, researchers, and innovators create, share, and refine machine learning models, datasets, and applications. With **600,000+ users** and **200,000+ contributors**, the platform hosts **2M+ models**, **1M+ applications**, and **500k+ datasets**, making it a cornerstone for open-source AI development.  

### **Our Mission**  
To democratize AI by empowering everyone to build, test, and deploy machine learning solutions—regardless of expertise level.  

---

## 🛠️ **What We Offer**  
### **For Developers & Researchers**  
- **Models**: Access or host AI models for text, image, video, audio, and more (e.g., Qwen3.5, FLUX, DLSS).  
- **Datasets**: Share or explore curated datasets for NLP, computer vision, and multimodal tasks.  
- **Spaces**: Build and deploy **AI apps** (e.g., video generation, image editing, text-to-video).  
- **Buckets**: Collaborate with pre-trained models in a controlled environment.  

### **For Teams & Enterprises**  
- **Enterprise Plans**: Secure access to AI tools with SSO, audit logs, and regional data controls.  
- **Dedicated Support**: Enterprise-grade reliability and compliance for scalable AI workflows.  
- **Open Source Stack**: Accelerate development with the Hugging Face Transformers library and infrastructure.  

---

## 🌟 **Company Culture**  
- **Collaboration-First**: We thrive on open-source innovation and community co-creation.  
- **Remote-First**, global team with a culture of transparency and inclusivity.  
- **Passionate about education**: We host tutorials, workshops, and a robust documentation hub.  
- **Agile & Iterative**: Rapidly prototype and iterate with real-time feedback loops.  

---

## 🚀 **Why Partner with Us?**  
- **For Customers**: Cut time-to-market with pre-built models (e.g., **Qwen3.5-27B**, **FLUX.2-klein-9b**).  
- **For Investors**: Back the infrastructure of the AI revolution with 400% YoY growth.  
- **For Developers**: Open innovation with **0% cost to contribute** to public repositories.  
- **For Enterprises**: Scalable solutions with **enterprise-grade security** and SLA commitments.  

---

## 🎯 **Careers at Hugging Face**  
We are hiring globally for roles across **AI research**, **product development**, and **customer success**.  
- **Work with**: Cutting-edge LLMs (e.g., **Foundation-1**, **Nemotron**) and multimodal AI.  
- **Culture Values**: Autonomy, innovation, and impact.  
- **Benefits**: Remote flexibility, equity for key roles, and a 5-day workweek.  

**Explore careers**: [Careers Page](https://hugging.face/careers)  

---

## 📞 **Connect with Us**  
- **Website**: [huggingface.co](https://huggingface.co)  
- **Enterprise Inquiries**: [Contact Sales](https://huggingface.co/enterprise)  
- **Join the Community**: [GitHub](https://github.com/huggingface) | [Discord](discord.hugging.face)  

**Hugging Face**: *"The home of open-source AI."* 🤗  

---  

Let’s build the future of AI—together.

In [22]:
def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")